# TRACR Purdue Data Collection Demo

This notebook demonstrates METS-R controlled vehicles projected into CARLA for data collection on `PurdueWestLafayette_centered`. The demo surface combines the live METS-R Viz stream, a CARLA bird-eye tracking camera, ego-centered Kafka or Simu5G BSM records shown as SAE J2735 core fields, CARLA LiDAR points, and a vehicle-mounted CARLA camera from the same vehicle that carries the LiDAR sensor.

## Implementation proposal

The demo runs in METS-R authoritative projection mode: METS-R owns all vehicle movement, CARLA mirrors those vehicle locations for sensor collection, and the configured BSM stream exposes V2X messages: Kafka by default, or Simu5G through the OMNeT++ bridge. The helper in `tutorials/tracr_demo_support.py` keeps the notebook focused on presentation cells.

In [1]:
from pathlib import Path
import importlib
import os
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "tutorials":
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import tutorials.tracr_demo_support as tracr_demo_support
tracr_demo_support = importlib.reload(tracr_demo_support)

TRACRDashboard = tracr_demo_support.TRACRDashboard
launch_tracr_demo = tracr_demo_support.launch_tracr_demo
run_tracr_demo = tracr_demo_support.run_tracr_demo
benchmark_tracr_demo = tracr_demo_support.benchmark_tracr_demo
step_tracr_demo = tracr_demo_support.step_tracr_demo

print("Repository root:", REPO_ROOT)


Repository root: c:\Users\ALei\Documents\GitHub\METS-R_HPC


## Launch services

This starts METS-R in Docker, CARLA with the Purdue map, creates private trips, marks a subset as C-V2X BSM emitters, and starts the METS-R Viz WebSocket stream. The BSM panel is ego-centered: rows show the focused vehicle role plus SAE J2735 `coreData` fields (`msgCnt`, `secMark`, `lat`, `long`, `speed`, `heading`, etc.) and hide unrelated messages. Link latency/range are shown only as metadata because they are not BSM fields. It uses Kafka by default; set `bsm_stream_source="simu5g"` to route sender-to-ego BSMs through the Simu5G/OMNeT++ bridge on `127.0.0.1:9099`. Keep Docker Desktop host networking enabled before running this cell.

In [2]:
runtime = launch_tracr_demo(
    run_config="configs/run_cosim_CARLAPurdue.json",
    private_vehicle_count=200,
    v2x_vehicle_count=20,
    private_vehicle_start_id=1000,
    start_kafka=None,  # auto: Kafka only when bsm_stream_source="kafka"
    bsm_stream_source="kafka",  # "kafka" or "simu5g"
    simu5g_host="127.0.0.1",
    simu5g_port=9099,
    start_metsr=True,
    start_carla=True,
    carla_camera_z=100.0,
    bsm_poll_timeout_ms=1,
    bsm_max_records=120,
    projection_heading_smoothing=0.35,
)

runtime.viz_info

Connection established!
No registered METS-R client helper servers found.
METS-R Vis live stream is available at ws://127.0.0.1:8766; origin=(-86.0, 40.0); call render() to send frames.
METS-R Vis stream port 8765 was busy; using 8766 instead.


{'host': '0.0.0.0',
 'port': 8766,
 'url': 'ws://127.0.0.1:8766',
 'local_url': 'ws://127.0.0.1:8766',
 'localhost_url': 'ws://localhost:8766',
 'browser_url': 'ws://127.0.0.1:8766',
 'localhost_reachable': False,
 'manifest': {'format': 'metsr-trajectory-binary',
  'version': 8,
  'byteOrder': 'bigEndian',
  'chunkMagic': 'MRTB',
  'coordScale': 100000,
  'initialX': -86.0,
  'initialY': 40.0,
  'tickInterval': 1,
  'linkSnapshotInterval': 1,
  'chunkTickLimit': 1,
  'chunks': [],
  'activeChunk': None,
  'roadIdDictionary': ['-104',
   '-12073',
   '-12074',
   '-12075',
   '-12078',
   '-12079',
   '-12080',
   '-12081',
   '-12083',
   '-12084',
   '-12085',
   '-12086',
   '-12088',
   '-12089',
   '-12102',
   '-12125',
   '-12141',
   '-12145',
   '-12146',
   '-12147',
   '-12148',
   '-12149',
   '-12150',
   '-12151',
   '-12152',
   '-12153',
   '-12154',
   '-12191',
   '-12192',
   '-12194',
   '-12195',
   '-12196',
   '-12197',
   '-12200',
   '-12201',
   '-12202',
   '

## Demo dashboard

Run this cell to start a local browser dashboard. Open the printed `http://127.0.0.1:8899/index.html` URL in Chrome or Edge. The dashboard embeds the hosted Purdue METS-R Vis page with local-network iframe permission so icons and the Mapbox basemap load from the normal deployment. If the embedded **Stream** still fails, use the **open top-level** link in the METS-R panel, paste the exact WebSocket URL shown below the panel, then click **Stream** there. Press `F11` for true full screen.

In [ ]:
dashboard = TRACRDashboard(
    stream_url=runtime.viz_info["url"],
    fullscreen=True,
    bsm_stream_label=runtime.bsm_stream_label,
    bsm_ego_only=True,
)
dashboard_url = dashboard.display_external(port=8899)
dashboard_url

Serving c:\Users\ALei\Documents\GitHub\METS-R_HPC\output\tracr_dashboard with CORS enabled on port 8899...


'http://127.0.0.1:8899/index.html'

127.0.0.1 - - [01/Jul/2026 23:26:59] "GET /state.json?ts=1782962818202 HTTP/1.1" 200 -
127.0.0.1 - - [01/Jul/2026 23:26:59] "GET /state.json?ts=1782962819213 HTTP/1.1" 200 -
127.0.0.1 - - [01/Jul/2026 23:26:59] "GET /state.json?ts=1782962818702 HTTP/1.1" 200 -
127.0.0.1 - - [01/Jul/2026 23:26:59] "GET /state.json?ts=1782962819711 HTTP/1.1" 200 -
127.0.0.1 - - [01/Jul/2026 23:27:00] "GET /state.json?ts=1782962820207 HTTP/1.1" 200 -
127.0.0.1 - - [01/Jul/2026 23:27:00] "GET /state.json?ts=1782962820703 HTTP/1.1" 200 -
127.0.0.1 - - [01/Jul/2026 23:27:01] "GET /state.json?ts=1782962821207 HTTP/1.1" 200 -
127.0.0.1 - - [01/Jul/2026 23:27:01] "GET /state.json?ts=1782962821700 HTTP/1.1" 200 -
127.0.0.1 - - [01/Jul/2026 23:27:02] "GET /state.json?ts=1782962822212 HTTP/1.1" 200 -
127.0.0.1 - - [01/Jul/2026 23:27:02] "GET /state.json?ts=1782962822708 HTTP/1.1" 200 -
127.0.0.1 - - [01/Jul/2026 23:27:03] "GET /state.json?ts=1782962823202 HTTP/1.1" 200 -
127.0.0.1 - - [01/Jul/2026 23:27:03] "GET /

## Run the live loop

Each iteration advances METS-R and CARLA one synchronized tick, mirrors METS-R private vehicles into CARLA, tracks the selected sensor vehicle in the CARLA bird-eye view, refreshes the latest vehicle-camera/LiDAR callbacks, filters the BSM panel to the current ego vehicle, and pushes frames to METS-R Viz. The demo loop intentionally throttles expensive side outputs: Kafka polling, dashboard redraws, and METS-R Viz rendering can run every N ticks while METS-R/CARLA synchronization still runs every tick.

In [ ]:
# Profile the live loop with presentation-friendly throttles.
last_result = benchmark_tracr_demo(
    runtime,
    dashboard,
    ticks=1500,
    sleep_s=0.0,
    render_wait_timeout=0,
    render_every=2,
    dashboard_every=3,
    bsm_every=3,
    sensor_every=1,
)
last_result["profile_summary_ms"]


{'bsm_stream': {'mean': 11.907472499297,
  'p50': 0.0,
  'p95': 62.931679969187826,
  'max': 126.99900002917275,
  'total': 7144.4834995782},
 'cosim_step': {'mean': 13.631044335197657,
  'p50': 12.519150011939928,
  'p95': 21.63744002173189,
  'max': 47.72410000441596,
  'total': 8178.626601118594},
 'dashboard_update': {'mean': 30.049693165929057,
  'p50': 0.0,
  'p95': 104.38247999118173,
  'max': 683.0161000252701,
  'total': 18029.815899557434},
 'deps': {'mean': 0.003468668437562883,
  'p50': 0.002500019036233425,
  'p95': 0.009399955160915852,
  'max': 0.023900007363408804,
  'total': 2.0812010625377297},
 'metsr_viz_render': {'mean': 13.849455167073756,
  'p50': 20.787599991308525,
  'p95': 38.412895024521255,
  'max': 99.09660002449527,
  'total': 8309.673100244254},
 'passive_carla': {'mean': 1.71650150034111,
  'p50': 1.1594999814406037,
  'p95': 3.3153300289995955,
  'max': 19.521200039889663,
  'total': 1029.900900204666},
 'road_projection': {'mean': 23.801779998660397,
 

## Manual step mode

Use this cell when you want precise control during a walkthrough.

In [ ]:
step_result = step_tracr_demo(runtime, dashboard=dashboard, render_wait_timeout=0, poll_bsm=True)
step_result


## Cleanup

Run this when the demo is done. Set `stop_kafka=True` if this notebook started Kafka and no other notebook needs it.

In [6]:
runtime.close(stop_kafka=False)